# Active learning with STACNotator + MLflow

This notebook runs the same active-learning round as
[`active-learning-demo.ipynb`](active-learning-demo.ipynb), but tracks every round as an
MLflow run:

- params, the held-out accuracy, and the trained model land in MLflow
- the prediction COG is attached to the run as an artifact
- the overlay registered in STACNotator carries `mlops_link` pointing at the run, so
  annotators can jump from the map layer straight to the experiment that produced it
- the run is tagged with the overlay name, closing the link in the other direction

Read the demo notebook first if the campaign/overlay/task-set flow is new; this one keeps
the explanations short and focuses on the MLflow wiring.

In [ ]:
# The SDK is not on PyPI yet; install it from this repo checkout
# (this notebook lives in sdk/examples/, the package root sdk/ is one dir up).
%pip install -q .. scikit-learn shapely mlflow

## 1. Point MLflow at a tracking server

`mlops_link` becomes a clickable link in the annotation UI, so the tracking URI must be an
HTTP server the annotators can open, not the default local `mlruns/` folder. For a local
try-out, start one next to this notebook:

```bash
mlflow ui --port 5000
```

In [ ]:
import mlflow

TRACKING_URI = "http://127.0.0.1:5000"  # or your team's MLflow server

mlflow.set_tracking_uri(TRACKING_URI)

## 2. Login, pick a campaign, and give it an experiment

One MLflow experiment per campaign keeps all its rounds comparable in one table.

In [ ]:
import stacnotator as snt

URL = "http://localhost:5173"  # your STACNotator app URL

snt.login(URL)
snt.campaigns()

In [ ]:
CAMPAIGN_ID = 91  # pick one from the table above

campaign = snt.campaign(CAMPAIGN_ID)
experiment = mlflow.set_experiment(f"stacnotator-campaign-{campaign.id}")
print(campaign)
print("labels:", campaign.labels)

## 3. Fetch samples and split

The test set is held out once and stays fixed, so the accuracy metric is comparable across
the runs of this experiment.

In [ ]:
from sklearn.model_selection import train_test_split

samples = campaign.get_samples()
train, test = train_test_split(samples, test_size=0.2, random_state=42)
print(f"{len(train)} train / {len(test)} test samples")

## 4. Train inside an MLflow run

The run is started here and stays open across the next few cells (train, predict, register,
queue) so everything from this round lands in one place; it is closed at the end of step 7.
As in the demo, lon/lat are the only features and polygon samples are reduced to their
centroid - swap in your real featurization here.

In [ ]:
import numpy as np
from shapely.geometry import shape
from sklearn.ensemble import RandomForestClassifier


def featurize(df):
    centroids = df["geometry"].map(lambda g: shape(g).centroid)
    lon = df["lon"].fillna(centroids.map(lambda c: c.x))
    lat = df["lat"].fillna(centroids.map(lambda c: c.y))
    return np.column_stack([lon, lat]), df["label_id"].to_numpy(dtype="int64")


ROUND = "round-1"
run = mlflow.start_run(run_name=ROUND)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(*featurize(train))
accuracy = model.score(*featurize(test))

mlflow.log_params({"n_estimators": 100, "n_train": len(train), "n_test": len(test)})
mlflow.log_metric("accuracy", accuracy)
mlflow.sklearn.log_model(model, name="model")
print(f"held-out accuracy: {accuracy:.2f}")

## 5. Predict over the extent and attach the COG to the run

Same dense pixel grid as the demo, including the noise nudge that breaks the toy model's
uniform sectors into landcover-looking patches (set `NOISE_STRENGTH = 0` for the raw model
map). `snt.utils.array_to_cog` turns the array plus the campaign extent into a valid
Cloud-Optimized GeoTIFF, and logging it as an artifact means the exact raster the annotators
saw is archived with the run that produced it.

In [ ]:
from scipy.ndimage import gaussian_filter

PIXEL_SIZE_DEG = 0.05  # make this smaller for a finer map

west, south, east, north = campaign.extent
width = max(2, int(np.ceil((east - west) / PIXEL_SIZE_DEG)))
height = max(2, int(np.ceil((north - south) / PIXEL_SIZE_DEG)))

lons = np.linspace(west, east, width)
lats = np.linspace(north, south, height)  # north up: first row is the top
grid = np.column_stack([np.tile(lons, height), np.repeat(lats, width)])

proba = model.predict_proba(grid).reshape(height, width, -1)


def landcover_noise(rng, octaves=((3, 0.2), (10, 0.5), (30, 1.0))):
    """Smooth random field mixing large blobs (sigma 30) with finer texture."""
    field = np.zeros((height, width))
    for sigma, weight in octaves:
        octave = gaussian_filter(rng.standard_normal((height, width)), sigma=sigma)
        field += weight * octave / octave.std()
    return field / field.std()


NOISE_STRENGTH = 0.5  # 0 -> raw model prediction
rng = np.random.default_rng(42)
noise = np.stack([landcover_noise(rng) for _ in model.classes_], axis=-1)
class_raster = model.classes_[np.argmax(proba + NOISE_STRENGTH * noise, axis=-1)].astype("uint8")

cog_path = snt.utils.array_to_cog(class_raster, campaign.extent, "predictions.cog.tif")
mlflow.log_param("pixel_size_deg", PIXEL_SIZE_DEG)
mlflow.log_artifact(str(cog_path))
print(f"wrote {cog_path} and attached it to the run")

## 6. Register the overlay, linked to the run

Upload `predictions.cog.tif` somewhere the tiler can reach (see the demo notebook for the
options) and put that URL in `COG_URL`. `mlops_link` shows up as a small link icon next to
the overlay selector in the annotation view, so annotators can jump straight to this run;
the tag on the run records which overlay this round produced.

In [ ]:
# The tiler fetches this URL, so it must NOT be a local file path.
COG_URL = "https://<your-storage>/predictions.cog.tif?<sas-token>"

run_url = f"{TRACKING_URI}/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}"
layer = campaign.register_overlay(
    COG_URL,
    name=f"predictions-{ROUND}",
    classes=campaign.labels,  # class values match label ids, legend shows label names
    mlops_link=run_url,
)
mlflow.set_tag("stacnotator.overlay", layer["name"])
mlflow.set_tag("stacnotator.campaign_id", campaign.id)
layer

## 7. Queue the most uncertain locations and close the run

Same uncertainty sampling as the demo, with the number of queued tasks logged before the
run is closed.

In [ ]:
import pandas as pd

N_QUERIES = 25  # how many locations to send to annotators this round

uncertainty = 1 - proba.reshape(len(grid), -1).max(axis=1)

most_uncertain = np.argsort(uncertainty)[::-1][:N_QUERIES]
queries = pd.DataFrame(
    {
        "lon": grid[most_uncertain, 0],
        "lat": grid[most_uncertain, 1],
        "uncertainty": uncertainty[most_uncertain].round(3),
    }
)

created = campaign.upload_tasks(queries, task_set=f"uncertain-{ROUND}", create_missing=True)
mlflow.log_metric("queued_tasks", created)
mlflow.end_run()
print(f"queued {created} tasks in set 'uncertain-{ROUND}'")

## 8. Next round: grow the training set, retrain, compare

Once annotators have worked through the batch, `update_samples` grows the training set and
the next round is logged as its own run. The `with` form closes the run automatically since
this round is a single cell.

In [ ]:
train = campaign.update_samples(train, exclude=test)

with mlflow.start_run(run_name="round-2"):
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(*featurize(train))
    accuracy = model.score(*featurize(test))
    mlflow.log_params({"n_estimators": 100, "n_train": len(train), "n_test": len(test)})
    mlflow.log_metric("accuracy", accuracy)
    mlflow.sklearn.log_model(model, name="model")
print(f"held-out accuracy: {accuracy:.2f}")

In [ ]:
# Accuracy per round, straight from the tracking server.
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id], order_by=["attributes.start_time"]
)
runs[["tags.mlflow.runName", "metrics.accuracy", "params.n_train"]]

From here the loop repeats: each round is one MLflow run, one overlay (linked both ways),
and one uncertainty task set. The experiment page becomes the campaign's training history,
and every overlay in the annotation UI points back at the run that produced it.